# Chapter 24 – Worked examples

This notebook runs the examples of [Chapter 24](https://katra.ballardini.com.ar/power-grid-topology-optimization-course/0.2.00.2.0/part-4-optimization/ch24-evolutionary-algorithms/) live. The code lives in the modules next to it
(`genome_mutation.py`, `line_switching.py`, `switching_ga.py`, `annealing.py`); the cells only call them, so a
change in a module is a change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
# JupyterLite only: install the libraries this notebook and its chapter modules use.
%pip install -q colorama==0.4.6 deepdiff==8.6.2 geojson==3.3.0 matplotlib networkx numpy packaging pandas pandera==0.34.1 scipy tqdm typing_extensions tzdata
%pip install -q --no-deps pandapower==3.5.6

In [ ]:
import matplotlib

matplotlib.use('Agg')

import annealing
import genome_mutation
import line_switching
import matplotlib.pyplot as plt
import numpy as np
import switching_ga

## A padded, sorted genotype and one mutation

"Switch off at most $k$ of $n$ lines" as a sorted integer vector of length $k$ padded with the sentinel $n$.
`normalize` removes duplicates, sorts and pads, so equal sets have equal genotypes; `mutate` overwrites one slot
and normalizes again.

In [ ]:
rng = np.random.default_rng(seed=1)
n = 41
g = genome_mutation.normalize(np.array([7, 7, 41]), n)
print('normalized      :', g)
for _ in range(5):
    g = genome_mutation.mutate(g, n, rng)
    print('after a mutation:', g)

## The fitness: N-1 overload energy of IEEE 30 with lines switched off

`load_case30` scales every load and generator by 1.4 so that the unswitched grid has N-1 overloads.
`overload_energy(case, D)` rebuilds the PTDF and LODF of the grid without the lines in $D$ and sums the MW above
the limits over all non-bridge outages; an islanding set gets $+\infty$ (fitness $-\infty$).

In [ ]:
case = line_switching.load_case30()
print('E(empty)            :', round(line_switching.overload_energy(case), 3), 'MW')
print('E({L25-27})         :', round(line_switching.overload_energy(case, (34,)), 3), 'MW')
print('E({L24-25, L6-28})  :', round(line_switching.overload_energy(case, (32, 40)), 3), 'MW')
print('fitness({L12-13})   :', line_switching.fitness(case, (15,)), '(bus 13 is islanded)')
print('LODF vs power flow  :', f'{line_switching.dc_power_flow_check(case, (32, 40), 5):.1e}', 'MW')

## Brute force up to two lines

$1 + 41 + \binom{41}{2} = 862$ sets, enumerated lazily with `itertools.combinations` like ToOp's
`optimizer/dc_bruteforce/generator.py`.

In [ ]:
bf = line_switching.brute_force(case, 2)
best, value = bf.best
print(f'{bf.n_evaluations} evaluations in {bf.seconds:.2f} s, {bf.n_islanding} islanding sets')
print(f'optimum E = {-value:.3f} MW at {line_switching.describe(case, best)}')

## The genetic algorithm against that optimum

Three seeds with a budget of 300 evaluations, against the two-line optimum. The curve of best-so-far overload
energy shows how different the seeds are: Chapter 26 turns that spread into a statistical statement.

In [ ]:
runs = [switching_ga.genetic_algorithm(case, seed, budget=300) for seed in range(3)]
print(switching_ga.seed_table(runs, value, case).to_string())
fig, ax = plt.subplots(figsize=(6, 3.5))
switching_ga.plot_convergence(runs, value, ax)
plt.close(fig)

## Hill climbing stuck, annealing not

The toy landscape of Exercise 1: four lines, two slots; $\{0, 1\}$ scores 10, $\{2, 3\}$ scores 20, every mixed
pair 0. Every neighbor of $\{0, 1\}$ is worse, so hill climbing stays; annealing accepts a downhill step and
then climbs to the global optimum.

In [ ]:
stuck_value, escaped = annealing.toy_runs()
print('hill climbing ends at fitness', stuck_value, '; annealing reaches 20 in', escaped, 'of 20 runs')